# Bài 1
Đây là notebook chứa mã nguồn đầy đủ của bài 1.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import numpy as np
import pandas as pd
import pulp
import pyomo.environ as pyo
from scipy.optimize import linprog, minimize, milp, LinearConstraint, Bounds
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize as pymoo_minimize

from src.data_loader import get_data


In [ ]:
SCENARIOS = {
    'S1': {'desc': 'Truyền thống', 'alloc_weights': [0.60, 0.15, 0.10, 0.15], 'tfp_growth': 0.010, 'ai_adoption': 0.20},
    'S2': {'desc': 'Số hóa nhanh', 'alloc_weights': [0.30, 0.40, 0.15, 0.15], 'tfp_growth': 0.012, 'ai_adoption': 0.40},
    'S3': {'desc': 'AI dẫn dắt', 'alloc_weights': [0.20, 0.20, 0.40, 0.20], 'tfp_growth': 0.015, 'ai_adoption': 0.60},
    'S4': {'desc': 'Bao trùm số', 'alloc_weights': [0.40, 0.20, 0.10, 0.30], 'tfp_growth': 0.011, 'ai_adoption': 0.30},
    'S5': {'desc': 'Tối ưu cân bằng', 'alloc_weights': [0.35, 0.25, 0.20, 0.20], 'tfp_growth': 0.014, 'ai_adoption': 0.35},
}

In [ ]:
def solve_bai01(data_dir=None, alpha=0.33, beta=0.42, gamma=0.10, delta=0.08, theta=0.07, tfp_growth=0.012):
    # Vietnam macro data 2020-2025 (embedded - không phụ thuộc CSV)
    data = get_data(data_dir)
    years = data.macro_years
    Y  = data.macro_Y
    K  = data.macro_K
    L  = data.macro_L
    D  = data.macro_D
    AI = data.macro_AI
    H  = data.macro_H

    # Normalize exponents to sum=1
    s = alpha + beta + gamma + delta + theta or 1
    a, b, g, d, t = alpha/s, beta/s, gamma/s, delta/s, theta/s

    # TFP estimation
    A = Y / (K**a * L**b * D**g * AI**d * H**t)
    yhat = A.mean() * K**a * L**b * D**g * AI**d * H**t
    mape = float(np.mean(np.abs((Y - yhat) / Y)) * 100)

    # Forecast 2026-2030
    fy = np.arange(2026, 2031)
    Kf  = K[-1]  * (1.06) ** np.arange(1, 6)
    Lf  = L[-1]  * (1.06) ** np.arange(1, 6)
    Df  = np.linspace(D[-1],  30,  5)
    AIf = np.linspace(AI[-1], 100, 5)
    Hf  = np.linspace(H[-1],  35,  5)
    Af  = A[-1] * (1 + tfp_growth) ** np.arange(1, 6)
    Yf  = Af * Kf**a * Lf**b * Df**g * AIf**d * Hf**t

    # Growth decomposition
    contrib = {
        'K':   float(a * np.mean(np.diff(np.log(K)))),
        'L':   float(b * np.mean(np.diff(np.log(L)))),
        'D':   float(g * np.mean(np.diff(np.log(D)))),
        'AI':  float(d * np.mean(np.diff(np.log(AI)))),
        'H':   float(t * np.mean(np.diff(np.log(H)))),
        'TFP': float(np.mean(np.diff(np.log(A)))),
    }
    total = sum(contrib.values()) or 1
    contrib_pct = {k: round(v / total * 100, 2) for k, v in contrib.items()}

    # Scenarios
    Yf_high_tfp = (Yf * 1.05).tolist()
    Yf_ai_fast  = (Yf * 1.035).tolist()

    return {
        'years':         years.tolist(),
        'Y_actual':      Y.tolist(),
        'Y_hat':         yhat.tolist(),
        'A_t':           A.tolist(),
        'mape':          mape,
        'forecast_years': fy.tolist(),
        'forecast_series': Yf.tolist(),
        'forecast_high_tfp': Yf_high_tfp,
        'forecast_ai_fast':  Yf_ai_fast,
        'contrib_pct':   contrib_pct,
        'exponents':     {'alpha': a, 'beta': b, 'gamma': g, 'delta': d, 'theta': t},
        'gdp_2030':      float(Yf[-1]),
    }

In [ ]:
def _solve(B, min_I=25, min_AI=15, coef_I=0.85, coef_AI=1.20, coef_H=0.95, coef_RD=1.35):
    c = [-coef_I, -coef_AI, -coef_H, -coef_RD]
    # Ràng buộc:
    # 1. Tổng ngân sách <= B  => x1 + x2 + x3 + x4 <= B
    # 2. x2 + x4 >= 0.35 * (x1 + x2 + x3 + x4) 
    # => -0.35*x1 + 0.65*x2 - 0.35*x3 + 0.65*x4 >= 0
    # => 0.35*x1 - 0.65*x2 + 0.35*x3 - 0.65*x4 <= 0
    A_ub = [
        [1, 1, 1, 1],
        [0.35, -0.65, 0.35, -0.65]
    ]
    b_ub = [B, 0]
    bounds = [(min_I, None), (min_AI, None), (20, None), (10, None)]
    res = linprog(c, A_ub=A_ub, b_ub=b_ub, bounds=bounds, method='highs')
    if res.success:
        return res.x.tolist(), float(-res.fun), True
    return [0, 0, 0, 0], 0.0, False

In [ ]:
if __name__ == '__main__':
    # Test chạy hàm
    pass